# 특정 시 경계 지도 생성기

시 이름을 입력하면 해당 시의 경계만 1000dpi PNG 파일로 출력합니다.

## 기능
- 시 이름 입력 → 해당 시 경계만 그리기
- 구로 나뉜 시(수원시, 성남시, 고양시 등)는 시 단위로 병합
- 광역시 구 이름도 지원 (예: 강남구)
- 1000dpi 고해상도 PNG 출력

## 필요한 파일
- `skorea-municipalities-2018-geo.json` : 남한 시군구 경계 데이터

## 필요한 패키지
```
pip install matplotlib shapely numpy
```

In [ ]:
# 1. 패키지 임포트 및 설정
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from shapely.geometry import shape
from shapely.ops import unary_union

# 한글 폰트
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

print("설정 완료")

In [ ]:
# 2. 파일 경로 및 설정값
DATA = r"skorea-municipalities-2018-geo.json"

# 출력 해상도
DPI = 1000

# 시 이름 입력 (여기서 변경)
TARGET_CITY = "수원시"

print(f"대상 시: {TARGET_CITY}")

In [ ]:
# 3. 데이터 로드 및 대상 시 찾기
with open(DATA, encoding="utf-8") as f:
    gj = json.load(f)

features = gj["features"]

# 대상 시에 해당하는 폴리곤 수집
target_geoms = []
matched_names = []

for ft in features:
    name = ft["properties"]["name"]
    geom = shape(ft["geometry"])

    # 정확히 일치하거나, 시 이름으로 시작하는 구(예: 수원시장안구) 포함
    if name == TARGET_CITY or name.startswith(TARGET_CITY):
        target_geoms.append(geom)
        matched_names.append(name)

if not target_geoms:
    raise ValueError(f"'{TARGET_CITY}'에 해당하는 시를 찾을 수 없습니다.")

# 병합 (구로 나뉜 시는 하나로 합침)
target = unary_union(target_geoms)

print(f"매칭된 구/군: {matched_names}")
print(f"병합 완료: {len(target_geoms)}개 폴리곤")

In [ ]:
# 4. 대상 시 경계 그리기
minx, miny, maxx, maxy = target.bounds

# 캔버스 크기 (시 크기에 비례, 최소 10인치)
w_deg = maxx - minx
h_deg = maxy - miny
base = 10.0
scale = max(w_deg, h_deg) / 1.0  # 1도 기준
W_INCH = max(base, base * w_deg)
H_INCH = max(base, base * h_deg)

fig, ax = plt.subplots(figsize=(W_INCH, H_INCH), dpi=DPI)
ax.set_aspect("equal")

# 배경 (바다)
ax.set_facecolor("#cfe8f7")
fig.set_facecolor("#cfe8f7")

# 대상 시 경계 채우기 + 경계선
if target.geom_type == "Polygon":
    polys_list = [target]
else:
    polys_list = list(target.geoms)

for poly in polys_list:
    x, y = poly.exterior.xy
    ax.fill(x, y, facecolor="#e8f4e8", edgecolor="#333333", linewidth=2.0, zorder=2)
    ax.plot(x, y, color="#333333", linewidth=2.0, zorder=3)

# 시 이름 표기
pt = target.representative_point()
ax.text(pt.x, pt.y, TARGET_CITY, fontsize=24, ha="center", va="center",
        color="#222222", zorder=5)

ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)
ax.axis("off")

plt.tight_layout(pad=0)
OUT = f"{TARGET_CITY}_map.png"
plt.savefig(OUT, dpi=DPI, facecolor=fig.get_facecolor())
print("saved:", OUT)

## 결과 확인

생성된 `{시이름}_map.png` 파일을 확인하세요.

## 설정 변경

2번 셀에서 다음 값을 변경할 수 있습니다:
- `TARGET_CITY` : 대상 시 이름 (예: "수원시", "강남구", "제주시")
- `DPI` : 해상도 (기본 1000)

## 지원하는 입력 예시
- `수원시` → 수원시 전체 (권선·영통·장안·팔달구 병합)
- `강남구` → 강남구만
- `제주시` → 제주시만
- `포항시` → 포항시 남구·북구 병합